> ⚠️ **Data-leakage warning (read before trusting the results).** The R² = 1.0 and 100% accuracy below are *not* real model skill. `Revenue Check` is defined as `Cost + Gross Profit`, which equals `Sales` exactly, and `Sales Category`/`Sales Per Unit` are derived from `Sales`. The 'High Demand' label is `Sales > mean(Sales)`, so predicting it from sales-derived columns is circular. A leak-free analysis is in `ml_leakage_free_check.py` and the README.

In [1]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================
# LOAD DATASET
# ============================================

df = pd.read_csv(
    r"feature_engineered_nassau_candy.csv"
)

# ============================================
# HANDLE MISSING VALUES
# ============================================

df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill numeric missing values
numeric_columns = df.select_dtypes(include=np.number).columns
df[numeric_columns] = df[numeric_columns].fillna(
    df[numeric_columns].median()
)

# Fill text missing values
text_columns = df.select_dtypes(include=["object", "string"]).columns
df[text_columns] = df[text_columns].fillna("Unknown")

# ============================================
# ENCODE ALL CATEGORICAL COLUMNS
# ============================================

label_encoders = {}

categorical_columns = df.select_dtypes(include=["object", "string"]).columns

for col in categorical_columns:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col].astype(str))
    label_encoders[col] = encoder

# ============================================
# PREPARE FEATURES AND TARGET
# ============================================

target = "Sales"

drop_columns = [
    "Sales",
    "Order ID",
    "Product ID",
    "Order Date",
    "Ship Date"
]

# Drop only columns that actually exist
drop_columns = [col for col in drop_columns if col in df.columns]

X = df.drop(columns=drop_columns)
y = df[target]

# ============================================
# FINAL CHECK
# ============================================

print("Remaining text columns:")
print(X.select_dtypes(include=["object", "string"]).columns)

# ============================================
# TRAIN TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ============================================
# TRAIN MODEL
# ============================================

lr = LinearRegression()

lr.fit(X_train, y_train)

# ============================================
# PREDICTION
# ============================================

prediction = lr.predict(X_test)

# ============================================
# EVALUATION
# ============================================

print("\nLinear Regression Results")
print("-------------------------")
print("MAE :", mean_absolute_error(y_test, prediction))
print("RMSE:", np.sqrt(mean_squared_error(y_test, prediction)))
print("R² Score:", r2_score(y_test, prediction))

Remaining text columns:
Index([], dtype='str')

Linear Regression Results
-------------------------
MAE : 2.2522444919369776e-13
RMSE: 2.798341265554541e-13
R² Score: 1.0


In [2]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

# ============================================
# LOAD DATASET
# ============================================

df = pd.read_csv(
    r"feature_engineered_nassau_candy.csv"
)

# ============================================
# DATA CLEANING
# ============================================

df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill numeric missing values
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill text missing values
text_cols = df.select_dtypes(include=["object", "string"]).columns
df[text_cols] = df[text_cols].fillna("Unknown")

# ============================================
# LABEL ENCODING
# ============================================

label_encoders = {}

categorical_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in categorical_columns:

    encoder = LabelEncoder()

    df[col] = encoder.fit_transform(df[col].astype(str))

    label_encoders[col] = encoder

# ============================================
# SALES PREDICTION
# ============================================

drop_columns = [
    "Sales",
    "Order ID",
    "Product ID",
    "Order Date",
    "Ship Date"
]

drop_columns = [
    col for col in drop_columns if col in df.columns
]

X = df.drop(columns=drop_columns)

y = df["Sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Linear Regression

lr = LinearRegression()

lr.fit(X_train, y_train)

lr_prediction = lr.predict(X_test)

print("\n========== SALES PREDICTION ==========")

print("Linear Regression")

print("MAE :", mean_absolute_error(y_test, lr_prediction))

print("RMSE :", np.sqrt(mean_squared_error(y_test, lr_prediction)))

print("R² Score :", r2_score(y_test, lr_prediction))

# Random Forest

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

rf_prediction = rf.predict(X_test)

print("\nRandom Forest")

print("MAE :", mean_absolute_error(y_test, rf_prediction))

print("RMSE :", np.sqrt(mean_squared_error(y_test, rf_prediction)))

print("R² Score :", r2_score(y_test, rf_prediction))

# ============================================
# FEATURE IMPORTANCE
# ============================================

importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": rf.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

print("\n========== TOP VARIABLES AFFECTING SALES ==========")

print(importance.head(10))

# ============================================
# DEMAND PREDICTION
# ============================================

if "High Demand" in df.columns:

    X = df.drop(
        columns=[
            "High Demand",
            "Order ID",
            "Product ID",
            "Order Date",
            "Ship Date"
        ],
        errors="ignore"
    )

    y = df["High Demand"]

    X_train, X_test, y_train, y_test = train_test_split(

        X,

        y,

        test_size=0.20,

        random_state=42

    )

    classifier = RandomForestClassifier(

        n_estimators=200,

        random_state=42

    )

    classifier.fit(X_train, y_train)

    demand_prediction = classifier.predict(X_test)

    print("\n========== DEMAND PREDICTION ==========")

    print("Accuracy :", accuracy_score(y_test, demand_prediction))

    print("\nConfusion Matrix")

    print(confusion_matrix(y_test, demand_prediction))

    print("\nClassification Report")

    print(classification_report(y_test, demand_prediction))

# ============================================
# PROFIT PREDICTION
# ============================================

if "Gross Profit" in df.columns:

    X = df.drop(
        columns=[
            "Gross Profit",
            "Order ID",
            "Product ID",
            "Order Date",
            "Ship Date"
        ],
        errors="ignore"
    )

    y = df["Gross Profit"]

    X_train, X_test, y_train, y_test = train_test_split(

        X,

        y,

        test_size=0.20,

        random_state=42

    )

    profit_model = RandomForestRegressor(

        n_estimators=200,

        random_state=42

    )

    profit_model.fit(X_train, y_train)

    profit_prediction = profit_model.predict(X_test)

    print("\n========== PROFIT PREDICTION ==========")

    print("MAE :", mean_absolute_error(y_test, profit_prediction))

    print("RMSE :", np.sqrt(mean_squared_error(y_test, profit_prediction)))

    print("R² Score :", r2_score(y_test, profit_prediction))

# ============================================
# SAVE LABEL ENCODERS
# ============================================

joblib.dump(
    label_encoders,
    r"label_encoders.pkl"
)

print("\nLabel Encoders Saved Successfully.")


========== SALES PREDICTION ==========
Linear Regression
MAE : 2.2522444919369776e-13
RMSE : 2.798341265554541e-13
R² Score : 1.0

Random Forest
MAE : 0.00734104953410335
RMSE : 0.17773385664647712
R² Score : 0.9996982750277075

========== TOP VARIABLES AFFECTING SALES ==========
            Feature  Importance
30    Revenue Check    0.707078
12             Cost    0.248440
11     Gross Profit    0.020158
10            Units    0.016670
26   Sales Category    0.002696
27  Profit Category    0.001001
5    State/Province    0.000370
23  Profit Per Unit    0.000295
6       Postal Code    0.000281
22    Cost Per Unit    0.000274

========== DEMAND PREDICTION ==========
Accuracy : 1.0

Confusion Matrix
[[1227    0]
 [   0  812]]

Classification Report
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      1227
        True       1.00      1.00      1.00       812

    accuracy                           1.00      2039
   macro avg       1.00 